<a href="https://colab.research.google.com/github/Permurez/Heartbeat/blob/main/notebooks/03_presentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Heart rate from video: remote photoplethysmography (rPPG)

**Grafika i GUI (G2UI)** semester project — Wroclaw University of Science and Technology.

This notebook walks through a complete rPPG pipeline end to end, on one recording from the
**UBFC-rPPG** dataset (Bobbia et al., *Pattern Recognition Letters*, 2019):

1. Why skin colour pulses with the heartbeat
2. Loading a recording from UBFC-rPPG
3. Face detection and the measurement region (ROI)
4. The raw colour signal over time
5. Filtering: isolating the heart-rate band
6. From colour to pulse: GREEN, CHROM, POS (with the maths behind each)
7. Frequency spectrum and reading off the BPM, checked against the reference sensor
8. The "wow" effect: making the pulse visible with Eulerian Video Magnification

All production code lives in the `rppg/` package of this repository; this notebook imports
it rather than re-implementing it, and shows the source of the key functions next to the
maths that explains them, so every formula below is tied to runnable code.

Repository: https://github.com/Permurez/Heartbeat


## 1. Why does skin colour pulse with the heartbeat?

Every heartbeat pushes a pulse of blood into the capillaries just under the skin. Haemoglobin
in that blood absorbs light, so the *amount* of blood present at any instant slightly changes
how much light the skin reflects back to a camera. More blood -> more absorption -> slightly
darker skin; less blood -> slightly brighter skin. This is the same physical effect a
fingertip pulse oximeter measures (through *transmitted* light), just observed through
*reflected* light at the face instead — which is why this family of techniques is called
**remote** photoplethysmography (rPPG), as opposed to the usual (contact) PPG.

A simple Beer–Lambert-style model for the reflected light intensity $I$ at wavelength
$\lambda$, reaching a camera pixel over a skin region, is

$$
I(\lambda, t) \;\propto\; I_0(\lambda)\, \exp\!\big(-\epsilon(\lambda)\, c(t)\, \ell(t)\big)
$$

where $c(t)$ is the blood volume/concentration pulsing with the heart rate, $\ell(t)$ is the
optical path length through the tissue, and $\epsilon(\lambda)$ is haemoglobin's
wavelength-dependent absorption coefficient. Two consequences matter for everything that
follows:

* $\epsilon(\lambda)$ peaks in the green part of the visible spectrum (~500–580 nm), so the
  pulse-related variation is largest in the **green** camera channel — the basis of the
  GREEN method below (Verkruysse, Svaasand & Nelson, 2008).
* The pulsatile change is tiny: on ordinary video, $c(t)$ varies the pixel intensity by well
  under 1%. The naked eye cannot see it — section 8 makes it visible by *amplifying* exactly
  this term with Eulerian Video Magnification (Wu et al., 2012).

**References**
- Verkruysse, Svaasand & Nelson, "Remote plethysmographic imaging using ambient light",
  *Optics Express*, 2008.
- de Haan & Jeanne, "Robust Pulse Rate From Chrominance-Based rPPG", *IEEE TBME*, 2013.
- Wang, den Brinker, Stuijk & de Haan, "Algorithmic Principles of Remote PPG", *IEEE TBME*, 2017.
- Wu, Rubinstein, Shih, Guttag, Durand & Freeman, "Eulerian Video Magnification for Revealing
  Subtle Changes in the World", *SIGGRAPH*, 2012.
- Bobbia, Macwan, Benezeth, Mansouri & Dubois, "Unsupervised skin tissue segmentation for
  remote photoplethysmography", *Pattern Recognition Letters*, 2019. (UBFC-rPPG dataset)


## 2. Setup

Mounts the dataset's Google Drive folder, clones this repository (or pulls the latest
version if it is already there) and installs MediaPipe, the face-landmark model used for
section 3. `ubfc_rppg.zip` must already be at `Config.zip_path` on Drive — the 32 GB file
itself is never committed to the repository (see `.gitignore`).


In [ ]:
# === SETUP ===
from google.colab import drive
drive.mount('/content/drive')

REPO_DIR = '/content/Heartbeat'
!git clone -q https://github.com/Permurez/Heartbeat.git {REPO_DIR} || git -C {REPO_DIR} pull -q
!pip install -q "mediapipe>=0.10.20"

import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import inspect
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Video

import rppg.ubfc as ubfc
import rppg.roi as roi
import rppg.pipeline as pipeline
import rppg.demo as demo
from rppg import Config
from rppg.groundtruth import bandpass, hr_spectral, hr_peaks
from rppg.methods import green, chrom, pos

cfg = Config()
SUBJECT = 'subject1'
cfg


## 3. Loading a recording from UBFC-rPPG

The dataset stores one folder per subject inside a single zip: `subjectN/vid.avi` (the face
video) and `subjectN/ground_truth.txt` (three rows: the reference PPG waveform, the oximeter's
own BPM reading, and frame timestamps in seconds). `rppg.ubfc` reads both without ever
unpacking the whole 32 GB archive at once.


In [ ]:
subjects = ubfc.list_subjects(cfg.zip_path)
print(f"{len(subjects)} subjects, e.g. {subjects[:5]}")

ppg, hr_oximeter, t = ubfc.load_ground_truth(cfg.zip_path, SUBJECT)
w, h, fps, n_frames = None, None, None, None
with ubfc.extracted_video(cfg.zip_path, SUBJECT, cfg.tmp_dir) as video_path:
    w, h, fps, n_frames = roi.probe_video(video_path)
    first_frame = next(roi.iter_frames_rgb(video_path, w, h))

print(f"Video: {w}x{h}, {fps:.2f} FPS, {n_frames} frames ({n_frames / fps:.1f} s)")
print(f"Ground truth: {len(t)} samples, {t[-1]:.1f} s, oximeter HR {hr_oximeter.min():.0f}-{hr_oximeter.max():.0f} BPM")

plt.figure(figsize=(5, 4))
plt.imshow(first_frame)
plt.title(f'{SUBJECT}, frame 0')
plt.axis('off')
plt.show()


## 4. Face detection and the measurement region (ROI)

A region of interest (ROI, the patch of skin a signal is measured from) must (a) actually be
skin, (b) stay on the same body part as the face moves, and (c) avoid eyes, eyebrows, lips and
hairline, which do not carry the same optical signal. `rppg.roi` uses MediaPipe's Face
Landmarker (478 points per frame) and builds three ROIs — forehead, left cheek, right cheek —
as the convex hull of a fixed subset of landmarks, shrunk slightly toward their centre to stay
clear of edges.

`pipeline.build_cache` runs this (and the per-frame colour statistics of section 5) over an
entire video once and caches the result; it is **resumable**, so a Colab disconnect only costs
the one subject that was in progress. Running it here on one subject also saves a debug image
with the ROIs drawn on it, worth checking visually before trusting any downstream numbers.


In [ ]:
summary = pipeline.build_cache(cfg, subjects=[SUBJECT])
display(summary)

debug_img_path = f'{cfg.cache_dir}/debug/{SUBJECT}.png'
plt.figure(figsize=(6, 6))
plt.imshow(plt.imread(debug_img_path))
plt.title('Face landmarks (MediaPipe) and skin ROIs (green)')
plt.axis('off')
plt.show()


## 5. The raw colour signal over time

For every frame, each ROI contributes a mean R, G, B value. Camera frames do not arrive at
perfectly regular intervals (jitter, occasionally a dropped or duplicated frame — see
`rppg.timeseries.clean_timestamps`), so the raw per-frame trace is first resampled onto a
uniform 30 Hz time grid (`pipeline.resample_subject`) before any further processing.

Plotted directly, the raw signal looks like noise: the pulse-related change is on the order of
a fraction of a percent of the mean brightness (section 1), buried under sensor noise, tiny
head motion and ambient lighting changes. This is exactly why every later step (filtering,
CHROM/POS) exists — to pull that one small periodic component out of everything else.


In [ ]:
raw = pipeline.load_subject(cfg, SUBJECT)
res = pipeline.resample_subject(raw, cfg)

roi_names = ['forehead', 'left cheek', 'right cheek']
window = (res['timestamps_s'] >= 20) & (res['timestamps_s'] < 30)       # a representative 10 s excerpt

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5), sharey=True)
for i, ax in enumerate(axes):
    for c, colour in enumerate('rgb'):
        ax.plot(res['timestamps_s'][window], res['rgb_raw'][window, i, c], color=colour, lw=0.8)
    ax.set_title(roi_names[i])
    ax.set_xlabel('time [s]')
axes[0].set_ylabel('mean pixel value [0-255]')
fig.suptitle('Raw ROI colour traces: the pulse is invisible by eye in the raw signal')
plt.tight_layout()
plt.show()


## 6. Filtering: isolating the heart-rate band

A plausible human heart rate (`Config.heart_rate_min_bpm`-`Config.heart_rate_max_bpm`, here 42-180 BPM) maps
to a narrow frequency band:

$$
f_{lo} = \frac{HR_{min}}{60}\ \text{Hz}, \qquad f_{hi} = \frac{HR_{max}}{60}\ \text{Hz}
$$

Everything outside $[f_{lo}, f_{hi}]$ — slow lighting drift, DC brightness, high-frequency
sensor noise — is discarded with a **Butterworth band-pass filter**. An order-$n$ Butterworth
low-pass has the maximally flat magnitude response

$$
|H(f)|^2 = \frac{1}{1 + (f / f_c)^{2n}}
$$

(no ripple in the passband, at the cost of a more gradual roll-off than other filter
families); a band-pass of order $n$ combines a high-pass and low-pass at $f_{lo}$ and $f_{hi}$.
Critically, the filter is applied with **`scipy.signal.filtfilt`**, which runs it forward and
then backward in time. This squares the magnitude response (sharper cutoff) and, by
construction, cancels any phase shift — so a peak in the filtered signal stays at the same
time as the corresponding heartbeat, which matters for beat-counting HR estimates later.


In [ ]:
print(inspect.getsource(bandpass))

green_raw = res['rgb_raw'][:, :, 1].mean(axis=1)          # average green across the 3 ROIs
green_filtered = bandpass(green_raw, cfg.sample_rate_hz, cfg.heart_rate_min_bpm / 60, cfg.heart_rate_max_bpm / 60)

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
axes[0].plot(res['timestamps_s'][window], green_raw[window])
axes[0].set_title('Raw (average green channel)')
axes[1].plot(res['timestamps_s'][window], green_filtered[window], color='tab:green')
axes[1].set_title(f'Band-pass filtered ({cfg.heart_rate_min_bpm:.0f}-{cfg.heart_rate_max_bpm:.0f} BPM) — the pulse is now visible')
axes[1].set_xlabel('time [s]')
plt.tight_layout()
plt.show()


## 7. From colour to pulse: GREEN, CHROM, POS

Filtering alone removes *frequencies* outside the heart-rate band, but motion and lighting
artefacts often fall **inside** that band too (someone shifting slightly, a flicker at a
similar rate to the pulse). The three methods below differ in how much they additionally
protect against that.

### GREEN

The simplest method: just use the green channel (section 1 explains why it carries the
strongest pulse signal). It has no defence against in-band brightness or motion artefacts —
anything that changes the green channel's brightness at a plausible heart-rate frequency is
indistinguishable from the real pulse.


In [ ]:
print(inspect.getsource(green))

green_rgb = res['rgb_lin'][:, :, :].mean(axis=1)           # (T, 3), averaged across ROIs
sig_green = bandpass(green(green_rgb), cfg.sample_rate_hz, cfg.heart_rate_min_bpm / 60, cfg.heart_rate_max_bpm / 60)


### CHROM (de Haan & Jeanne, 2013)

Normalise each channel by its own temporal mean (this removes the fixed skin-tone colour and
overall brightness level):

$$
R_n = \frac{R}{\bar R}, \qquad G_n = \frac{G}{\bar G}, \qquad B_n = \frac{B}{\bar B}
$$

and combine them into two "chrominance" signals, with coefficients chosen (de Haan & Jeanne,
from a standard skin-reflectance model) so both respond to the blood-volume pulse:

$$
X_s = 3R_n - 2G_n, \qquad Y_s = 1.5R_n + G_n - 1.5B_n
$$

Band-pass filter both to $X_f, Y_f$. A brightness or specular-reflection artefact that changes
every channel by the *same fraction* appears with an **identical** coefficient in both $X_f$
and $Y_f$ (check: the coefficients in $X_s$ sum to $3-2=1$, and in $Y_s$ to $1.5+1-1.5=1$) —
so when that artefact dominates, $X_f$ and $Y_f$ become near-identical copies of it, and a
scalar

$$
\alpha = \frac{\sigma(X_f)}{\sigma(Y_f)} \approx 1 \quad\text{(when the artefact dominates)}
$$

cancels it almost completely in

$$
S_{CHROM} = X_f - \alpha\, Y_f .
$$

When the pulse itself dominates instead, $\alpha$ takes a different value tuned to combine the
two chrominance signals' *pulse* content constructively. Either way, $\alpha$ is computed
fresh for every window, adapting automatically to what is currently the dominant source of
variation.


In [ ]:
print(inspect.getsource(chrom))

sig_chrom = chrom(green_rgb, cfg.sample_rate_hz, cfg.heart_rate_min_bpm / 60, cfg.heart_rate_max_bpm / 60)


### POS (Wang, den Brinker, Stuijk & de Haan, 2017)

POS starts from the same goal but derives the projection geometrically instead of empirically.
In normalised RGB space, a brightness or specular change that affects every channel by the
*same fraction* moves the signal exactly along the direction $(1, 1, 1)$ — so the pulse, which
does *not* behave that way (section 1: it affects R, G, B by different, wavelength-dependent
amounts), must live in the **plane orthogonal to $(1,1,1)$**.

For a short sliding window (the paper uses $\approx 1.6$ s) of normalised RGB $C_n(t) =
C(t) / \bar C$ (temporal mean over that window only):

$$
S_1 = G_n - B_n, \qquad S_2 = -2R_n + G_n + B_n
$$

Unlike CHROM's $X_s, Y_s$, the coefficients of $S_1$ and $S_2$ **each sum to zero**
($1-1=0$ and $-2+1+1=0$): any common fractional change in all three channels is cancelled
*exactly*, by construction, with no reliance on an adaptive ratio. The two in-plane axes are
then combined the same way as CHROM, with a ratio that equalises their amplitude:

$$
\alpha = \frac{\sigma(S_1)}{\sigma(S_2)}, \qquad h = S_1 + \alpha\, S_2
$$

Each window's $h$ is de-meaned and added into the output at its own time position; overlapping
windows reinforce the shared pulse component and smooth out residual drift, acting like an
adaptive filter without an explicit cutoff frequency.


In [ ]:
print(inspect.getsource(pos))

sig_pos = pos(green_rgb, cfg.sample_rate_hz)
sig_pos = bandpass(sig_pos, cfg.sample_rate_hz, cfg.heart_rate_min_bpm / 60, cfg.heart_rate_max_bpm / 60)


### Side by side

All three signals, normalised to unit standard deviation so they can be compared on one axis,
over the same 10 s excerpt used above.


In [ ]:
def norm(x):
    return (x - x[window].mean()) / x[window].std()

fig, ax = plt.subplots(figsize=(12, 4))
for sig, label, colour in [(sig_green, 'GREEN', 'tab:green'),
                           (sig_chrom, 'CHROM', 'tab:orange'),
                           (sig_pos, 'POS', 'tab:purple')]:
    ax.plot(res['timestamps_s'][window], norm(sig)[window], label=label, color=colour, alpha=0.85)
ax.set_xlabel('time [s]')
ax.set_title('GREEN vs CHROM vs POS: the same pulse, three ways of extracting it')
ax.legend()
plt.tight_layout()
plt.show()


## 8. Frequency spectrum and reading off the BPM

The pulse signal is (quasi-)periodic, so its heart rate shows up as a sharp peak in its
frequency spectrum. The discrete Fourier transform of a windowed signal $x[0..N-1]$ is

$$
\hat X[k] = \sum_{m=0}^{N-1} x[m]\, w[m]\, e^{-i 2\pi k m / N}, \qquad f_k = \frac{k f_s}{N}
$$

where $w[m]$ is a Hann window (tapers the signal at the edges, reducing spectral leakage from
treating a finite excerpt as if it were periodic). Zero-padding to a longer length $N_{FFT}$
adds no information — it only samples the same underlying spectrum more densely, so the peak's
frequency can be read off with sub-BPM precision. Restricting the search to the plausible
heart-rate band and taking the strongest peak there gives the estimate:

$$
\widehat{HR} = 60 \cdot \operatorname*{arg\,max}_{f \in [f_{lo},\, f_{hi}]} \big|\hat X(f)\big|
$$

exactly what `rppg.groundtruth.hr_spectral` computes (used throughout this project, including
for the *reference* HR — cross-checked against a beat-counting estimate, see README).


In [ ]:
print(inspect.getsource(hr_spectral))

def spectrum(signal, sample_rate_hz, fft_length=2**14):
    from scipy.signal import detrend
    signal = detrend(np.asarray(signal, float))
    spec = np.abs(np.fft.rfft(signal * np.hanning(len(signal)), n=fft_length))
    freqs = np.fft.rfftfreq(fft_length, d=1.0 / sample_rate_hz)
    return freqs * 60.0, spec                         # x-axis in BPM

W = int(cfg.window_s * cfg.sample_rate_hz)
start = int(20 * cfg.sample_rate_hz)                               # the same 8 s window used elsewhere
sl = slice(start, start + W)

ppg_filt = bandpass(res['ppg'][sl], cfg.sample_rate_hz, cfg.heart_rate_min_bpm / 60, cfg.heart_rate_max_bpm / 60)
hr_ref = hr_spectral(ppg_filt, cfg.sample_rate_hz, cfg.heart_rate_min_bpm, cfg.heart_rate_max_bpm)

fig, ax = plt.subplots(figsize=(10, 4))
for sig, label, colour in [(sig_green[sl], 'GREEN', 'tab:green'),
                           (sig_chrom[sl], 'CHROM', 'tab:orange'),
                           (sig_pos[sl], 'POS', 'tab:purple')]:
    bpm, spec = spectrum(sig, cfg.sample_rate_hz)
    hr_est = hr_spectral(sig, cfg.sample_rate_hz, cfg.heart_rate_min_bpm, cfg.heart_rate_max_bpm)
    ax.plot(bpm, spec / spec.max(), label=f'{label} ({hr_est:.1f} BPM)', color=colour)
ax.axvline(hr_ref, color='k', ls='--', label=f'reference ({hr_ref:.1f} BPM)')
ax.set_xlim(cfg.heart_rate_min_bpm, cfg.heart_rate_max_bpm)
ax.set_xlabel('heart rate [BPM]')
ax.set_ylabel('normalised amplitude')
ax.set_title('Frequency spectrum of one 8 s window: each method vs the reference sensor')
ax.legend()
plt.tight_layout()
plt.show()


### Heart rate over the whole recording

The same spectral estimate, repeated for every 8 s window (stepping every second) across the
full recording, compared against the reference oximeter signal — this is the headline result
of the whole pipeline.


In [ ]:
import pandas as pd

def hr_track(sig):
    out = []
    for s in range(0, len(sig) - W + 1, int(cfg.sample_rate_hz)):
        out.append(hr_spectral(sig[s:s + W], cfg.sample_rate_hz, cfg.heart_rate_min_bpm, cfg.heart_rate_max_bpm))
    return np.array(out)

t_track = res['timestamps_s'][:len(res['timestamps_s']) - W + 1:int(cfg.sample_rate_hz)] + cfg.window_s / 2
ppg_full_filt = bandpass(res['ppg'], cfg.sample_rate_hz, cfg.heart_rate_min_bpm / 60, cfg.heart_rate_max_bpm / 60)

tracks = {'reference': hr_track(ppg_full_filt), 'GREEN': hr_track(sig_green),
          'CHROM': hr_track(sig_chrom), 'POS': hr_track(sig_pos)}
hr_df = pd.DataFrame(tracks, index=t_track)
mae = {m: float((hr_df[m] - hr_df['reference']).abs().mean()) for m in ['GREEN', 'CHROM', 'POS']}

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(hr_df.index, hr_df['reference'], 'k-', lw=2.5, label='reference (oximeter)')
for m, colour in [('GREEN', 'tab:green'), ('CHROM', 'tab:orange'), ('POS', 'tab:purple')]:
    ax.plot(hr_df.index, hr_df[m], ':', lw=1.3, color=colour, label=f'{m} (MAE {mae[m]:.1f} BPM)')
ax.set_xlabel('time [s]')
ax.set_ylabel('heart rate [BPM]')
ax.set_title(f'{SUBJECT}: heart rate over time, camera vs reference')
ax.legend()
plt.tight_layout()
plt.show()
print('Mean absolute error vs reference [BPM]:', {k: round(v, 1) for k, v in mae.items()})


## 9. The "wow" effect: seeing the pulse (Eulerian Video Magnification)

Section 1 noted the pulse changes pixel intensity by well under 1% — invisible to the eye.
**Eulerian Video Magnification** (Wu et al., 2012) makes it visible without tracking any
pixels: it looks at every pixel's value *over time*, keeps only the frequencies in the
heart-rate band, and adds that back into the video multiplied by a gain $\alpha$.

**Colour magnification.** Writing a (spatially blurred, downsampled) pixel's intensity as a
baseline plus a small time-varying term, $I(x,t) = I_0(x) + \delta(x,t)$, band-pass filtering
$\delta$ in the heart-rate band and adding it back amplified reconstructs

$$
\hat I(x,t) = I(x,t) + \alpha\, \delta_{band}(x,t)
$$

— the same pixel, with its small periodic colour change inflated to something the eye can see.

**Motion magnification** applies the same idea to a **Laplacian pyramid** (a set of band-pass
*spatial* images) instead of raw pixels: small shifts of an edge show up as intensity changes
within each spatial band, and amplifying those (with the per-level gain capped, following Wu
et al., so that fine, noise-dominated levels are not blown up) makes small motions look larger
— here, a subtly pulsating face.

`rppg.demo.make_demo_video` renders both side by side with the ROI view, in a band centred on
the heart rate estimated **from the video itself** (not the reference — a real system would
not have that), reusing `rppg.evm.magnify_color` / `magnify_motion`.


In [ ]:
out_path = demo.make_demo_video(cfg, SUBJECT, start_s=20.0, duration_s=10.0)
Video(out_path, embed=True, width=900)


## 10. Summary and next steps

What this notebook showed, end to end on one UBFC-rPPG recording:

1. **Why** skin colour pulses with the heartbeat (optical absorption model).
2. **Loading** a recording and its reference sensor data from the dataset.
3. **Face detection and ROI** placement (MediaPipe landmarks, skin-only regions).
4. The **raw colour signal**, and why it looks like pure noise unprocessed.
5. **Filtering** to the heart-rate band with a zero-phase Butterworth band-pass.
6. Three ways to turn colour into a **pulse signal** — GREEN, CHROM, POS — with the maths
   that explains why CHROM and POS are more robust to motion/lighting than GREEN.
7. Reading **BPM** off the frequency spectrum, and checking it against ground truth over the
   whole recording.
8. Making the invisible pulse **visible** with Eulerian Video Magnification.

**Not yet done** (see the repository README for the up-to-date status):
- Fusing GREEN / CHROM / POS (and head-motion-based) estimates into one more robust estimate.
- A full evaluation across all subjects (MAE, RMSE, Pearson r, Bland–Altman) rather than one.
- Multi-person tracking.
- A live GUI (webcam input, per-face BPM, EVM preview).

Code: https://github.com/Permurez/Heartbeat — dataset: Bobbia et al., UBFC-rPPG, 2019.
